# Test and Validation Dataset Candidates Creation

In [2]:
from PyDI.io import load_parquet

# Define dataset paths
DATA_DIR = "../datasets/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "kaggle380k.parquet",
    name="kaggle380k",
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "uber_eats.parquet",
    name="uber_eats",
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "yelp.parquet",
    name="yelp",
)

In [20]:
# ==========================================
# Pairwise ground-truth creation (df1↔df2 and df1↔df3) — robust IDs & categories
# ==========================================
import math, re, ast
import pandas as pd
from difflib import SequenceMatcher

# ---------- Small helpers ----------
def _zip5(z):
    if pd.isna(z): return None
    m = re.search(r"(\d{5})", str(z))
    return m.group(1) if m else None

def _haversine_km(lat1, lon1, lat2, lon2):
    try:
        if None in (lat1, lon1, lat2, lon2):
            return float("inf")
        rlat1, rlon1, rlat2, rlon2 = map(math.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = rlat2 - rlat1, rlon2 - rlon1
        a = math.sin(dlat/2)**2 + math.cos(rlat1)*math.cos(rlat2)*math.sin(dlon/2)**2
        return 6371.0088 * (2 * math.asin(math.sqrt(a)))
    except Exception:
        return float("inf")

def _name_sim(a, b):
    if not a or not b: return 0.0
    return SequenceMatcher(None, str(a), str(b)).ratio()

def _to_set_of_str(x):
    """
    Coerce many possible 'categories' representations into a set[str] (lowercased, stripped).
    Handles: list/tuple/set, numpy arrays / pandas Series via .tolist(), JSON-like strings, CSV strings, scalars, None.
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return set()

    # Native container types
    if isinstance(x, (list, tuple, set)):
        return {str(v).strip().lower() for v in x if str(v).strip()}

    # Numpy arrays / pandas objects that expose .tolist()
    if hasattr(x, "tolist"):
        try:
            return {str(v).strip().lower() for v in x.tolist() if str(v).strip()}
        except Exception:
            pass

    # Strings
    if isinstance(x, str):
        s = x.strip()
        # Try Python/JSON literal first
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")) or (s.startswith("{") and s.endswith("}")):
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple, set)):
                    return {str(v).strip().lower() for v in obj if str(v).strip()}
                if isinstance(obj, dict):
                    return {str(k).strip().lower() for k, v in obj.items() if (v in (True, "True", "true", 1, "1"))}
            except Exception:
                pass
        # Fallback: split on common separators
        parts = re.split(r"[|,;/]+", s)
        return {p.strip().lower() for p in parts if p.strip()}

    # Scalar fallback
    s = str(x).strip().lower()
    return {s} if s else set()

def _jaccard_list(a, b):
    A = _to_set_of_str(a)
    B = _to_set_of_str(b)
    U = A | B
    if not U: return 0.0
    return len(A & B) / len(U)

def _last7(phone_e164):
    if not isinstance(phone_e164, str) or not phone_e164.strip():
        return None
    digits = re.sub(r"\D", "", phone_e164)
    return digits[-7:] if len(digits) >= 7 else None

def _ensure_id_cols_after_merge(cand: pd.DataFrame, id_col_a: str, id_col_b: str):
    """
    After a merge with suffixes (_a/_b), create stable 'id_a'/'id_b'.
    Falls back to unsuffixed columns or row index if needed.
    """
    ida_suf, idb_suf = f"{id_col_a}_a", f"{id_col_b}_b"

    # id_a
    if ida_suf in cand.columns:
        cand["id_a"] = cand[ida_suf]
    elif id_col_a in cand.columns:
        cand["id_a"] = cand[id_col_a]
    elif "rowid_a" in cand.columns:
        cand["id_a"] = cand["rowid_a"]
    else:
        cand["id_a"] = cand.index

    # id_b
    if idb_suf in cand.columns:
        cand["id_b"] = cand[idb_suf]
    elif id_col_b in cand.columns:
        cand["id_b"] = cand[id_col_b]
    elif "rowid_b" in cand.columns:
        cand["id_b"] = cand["rowid_b"]
    else:
        cand["id_b"] = cand.index

    return cand

# ---------- Candidate builders (pairwise) ----------
def _cand_geo(dfA, dfB, id_col_a, id_col_b, decimals=3, max_km=0.2):
    Acols = ["source","name","name_norm","categories","latitude","longitude","street","house_number","city","state","postal_code", id_col_a]
    Bcols = ["source","name","name_norm","categories","latitude","longitude","street","house_number","city","state","postal_code", id_col_b]

    A = dfA.loc[dfA["latitude"].notna() & dfA["longitude"].notna(), Acols].copy()
    B = dfB.loc[dfB["latitude"].notna() & dfB["longitude"].notna(), Bcols].copy()
    if A.empty or B.empty: return pd.DataFrame()

    A["lat_r"] = A["latitude"].round(decimals); A["lon_r"] = A["longitude"].round(decimals)
    B["lat_r"] = B["latitude"].round(decimals); B["lon_r"] = B["longitude"].round(decimals)
    cand = A.merge(B, on=["lat_r","lon_r"], suffixes=("_a","_b"))
    if cand.empty: return cand

    dkm = cand.apply(lambda r: _haversine_km(r["latitude_a"], r["longitude_a"], r["latitude_b"], r["longitude_b"]), axis=1)
    cand = cand.loc[dkm <= max_km].copy()
    cand = cand.drop(columns=["lat_r","lon_r"])

    cand = _ensure_id_cols_after_merge(cand, id_col_a, id_col_b)
    cand["rule"] = "geo"
    return cand

def _cand_name_zip(dfA, dfB, id_col_a, id_col_b):
    A = dfA[["source","name","name_norm","postal_code","city","state","latitude","longitude","categories","street","house_number", id_col_a]].copy()
    B = dfB[["source","name","name_norm","postal_code","city","state","latitude","longitude","categories","street","house_number", id_col_b]].copy()
    if A.empty or B.empty: return pd.DataFrame()
    A["zip5"] = A["postal_code"].apply(_zip5)
    B["zip5"] = B["postal_code"].apply(_zip5)
    A["nz"] = A["name_norm"].str.replace(" ", "", regex=False) + "#" + A["zip5"].fillna("")
    B["nz"] = B["name_norm"].str.replace(" ", "", regex=False) + "#" + B["zip5"].fillna("")
    cand = A.merge(B, on="nz", suffixes=("_a","_b"))
    if cand.empty: return cand
    cand = cand.drop(columns=["nz"])
    cand = _ensure_id_cols_after_merge(cand, id_col_a, id_col_b)
    cand["rule"] = "name_zip"
    return cand

def _cand_address(dfA, dfB, id_col_a, id_col_b):
    cols = ["source","name","name_norm","categories","latitude","longitude","street","house_number","city","state","postal_code"]
    A0 = dfA[cols + [id_col_a]].copy().dropna(subset=["street","house_number"])
    B0 = dfB[cols + [id_col_b]].copy().dropna(subset=["street","house_number"])
    if A0.empty or B0.empty: return pd.DataFrame()

    A0["zip5"] = A0["postal_code"].apply(_zip5)
    B0["zip5"] = B0["postal_code"].apply(_zip5)

    # Path 1: with ZIP
    A_zip = A0.dropna(subset=["zip5"]); B_zip = B0.dropna(subset=["zip5"])
    p_zip = A_zip.merge(B_zip, on=["zip5","house_number","street"], suffixes=("_a","_b"), how="inner")

    # Path 2: with city/state
    A_cs = A0.dropna(subset=["city","state"]); B_cs = B0.dropna(subset=["city","state"])
    p_cs = A_cs.merge(B_cs, on=["city","state","house_number","street"], suffixes=("_a","_b"), how="inner")

    out = pd.concat([p_zip, p_cs], ignore_index=True) if (not p_zip.empty or not p_cs.empty) else pd.DataFrame()
    if out.empty: return out

    out = _ensure_id_cols_after_merge(out, id_col_a, id_col_b)
    out["rule"] = "addr_zip_or_citystate"
    return out

def _cand_phone(dfA, dfB, id_col_a, id_col_b):
    # Only contributes when BOTH have phone_e164
    if "phone_e164" not in dfA.columns or "phone_e164" not in dfB.columns:
        return pd.DataFrame()
    A = dfA[["source","name","name_norm","categories","latitude","longitude","phone_e164","street","house_number","city","state","postal_code", id_col_a]].copy()
    B = dfB[["source","name","name_norm","categories","latitude","longitude","phone_e164","street","house_number","city","state","postal_code", id_col_b]].copy()
    A["p7"] = A["phone_e164"].apply(_last7); B["p7"] = B["phone_e164"].apply(_last7)
    A = A.dropna(subset=["p7"]); B = B.dropna(subset=["p7"])
    if A.empty or B.empty: return pd.DataFrame()
    cand = A.merge(B, on="p7", suffixes=("_a","_b"))
    if cand.empty: return cand
    cand = cand.drop(columns=["p7"])
    cand = _ensure_id_cols_after_merge(cand, id_col_a, id_col_b)
    cand["rule"] = "phone"
    return cand

def _cand_city_name(dfA, dfB, id_col_a, id_col_b):
    A = dfA[["source","name","name_norm","categories","latitude","longitude","city","state","street","house_number","postal_code", id_col_a]].dropna(subset=["city","state","name_norm"]).copy()
    B = dfB[["source","name","name_norm","categories","latitude","longitude","city","state","street","house_number","postal_code", id_col_b]].dropna(subset=["city","state","name_norm"]).copy()
    if A.empty or B.empty: return pd.DataFrame()
    A["csn"] = A["city"].str.lower().str.strip() + "|" + A["state"].str.upper().str.strip() + "|" + A["name_norm"].str.strip()
    B["csn"] = B["city"].str.lower().str.strip() + "|" + B["state"].str.upper().str.strip() + "|" + B["name_norm"].str.strip()
    cand = A.merge(B, on="csn", suffixes=("_a","_b"))
    if cand.empty: return cand
    cand = cand.drop(columns=["csn"])
    cand = _ensure_id_cols_after_merge(cand, id_col_a, id_col_b)
    cand["rule"] = "city_name"
    return cand

# ---------- Scoring ----------
def _score_candidates(cand: pd.DataFrame):
    if cand.empty: return cand
    cand["sim_name"] = cand.apply(lambda r: _name_sim(r.get("name_norm_a"), r.get("name_norm_b")), axis=1)
    cand["same_house"] = (cand.get("house_number_a") == cand.get("house_number_b"))
    cand["same_street"] = (cand.get("street_a") == cand.get("street_b"))
    cand["zip5_a"] = cand.get("postal_code_a").apply(_zip5)
    cand["zip5_b"] = cand.get("postal_code_b").apply(_zip5)
    cand["same_zip5"] = (cand["zip5_a"] == cand["zip5_b"])
    cand["same_city_state"] = (cand.get("city_a").str.lower().fillna("") == cand.get("city_b").str.lower().fillna("")) & \
                              (cand.get("state_a").str.upper().fillna("") == cand.get("state_b").str.upper().fillna(""))
    cand["geo_km"] = cand.apply(lambda r: _haversine_km(r.get("latitude_a"), r.get("longitude_a"), r.get("latitude_b"), r.get("longitude_b")), axis=1)
    cand["geo_m"] = (cand["geo_km"] * 1000).round(1)

    # Robust categories similarity
    cand["sim_cat"] = cand.apply(lambda r: _jaccard_list(r.get("categories_a"), r.get("categories_b")), axis=1)

    # Buckets
    cand["bucket"] = "low"
    cand.loc[
        (cand["geo_m"] <= 50) |
        ((cand["same_house"] & cand["same_street"] & cand["same_zip5"]) & (cand["sim_name"] >= 0.85)) |
        (cand["sim_name"] >= 0.95),
        "bucket"
    ] = "high"
    cand.loc[
        (cand["bucket"] == "low") & (
            ((cand["geo_m"] <= 200) & (cand["sim_name"] >= 0.70)) |
            (cand["same_city_state"] & (cand["sim_name"] >= 0.88)) |
            (cand["same_house"] & cand["same_street"] & (cand["sim_name"] >= 0.70))
        ),
        "bucket"
    ] = "mid"

    cand["score"] = (
        (1.0 - cand["geo_km"].clip(upper=1.0)) * 0.40 +
        cand["sim_name"] * 0.40 +
        cand["sim_cat"] * 0.20
    )
    return cand

# ---------- Build candidates for a SINGLE pair ----------
def build_candidates_pair(dfA, dfB, pair_name: str,
                          id_col_a: str, id_col_b: str,
                          geo_decimals=3, geo_max_km=0.2):
    """
    Build union of candidate pairs for (A↔B) using several simple techniques,
    score them, and attach dataset-specific IDs as 'id_a'/'id_b'.
    """
    parts = []
    for fn in (_cand_geo, _cand_name_zip, _cand_address, _cand_phone, _cand_city_name):
        cand = fn(dfA, dfB, id_col_a=id_col_a, id_col_b=id_col_b,
                  decimals=geo_decimals, max_km=geo_max_km) if fn is _cand_geo \
               else fn(dfA, dfB, id_col_a=id_col_a, id_col_b=id_col_b)
        if cand is not None and not cand.empty:
            parts.append(cand)

    if not parts:
        return pd.DataFrame()

    C = pd.concat(parts, ignore_index=True)

    # Ensure id_a/id_b exist (safety)
    if "id_a" not in C.columns or "id_b" not in C.columns:
        C = _ensure_id_cols_after_merge(C, id_col_a, id_col_b)

    # De-duplicate by id pair when available
    subset = [c for c in ["id_a","id_b"] if c in C.columns]
    if subset:
        C = C.drop_duplicates(subset=subset)

    # Tag pair and score
    C["pair"] = pair_name
    C = _score_candidates(C)

    # Sort by score
    C = C.sort_values(["score"], ascending=False).reset_index(drop=True)
    return C

# ---------- Sampling & splitting (side-by-side layout) ----------
def sample_for_manual_labeling(cands: pd.DataFrame, n_total=800, seed=42,
                               id_col_a: str = None, id_col_b: str = None):
    """
    Sample ~20% likely matches (high), ~30% corner cases (mid), ~50% likely non-matches (low).
    Output columns are arranged SIDE-BY-SIDE for easy comparison (A vs B).
    """
    if cands.empty:
        raise ValueError("No candidates produced; cannot sample.")
    rng = pd.Series(range(len(cands))).sample(frac=1.0, random_state=seed).index
    C = cands.loc[rng].copy()

    n_pos   = max(1, int(round(n_total * 0.20)))
    n_corner= max(1, int(round(n_total * 0.30)))
    n_neg   = max(1, n_total - n_pos - n_corner)

    high = C[C["bucket"] == "high"]
    mid  = C[C["bucket"] == "mid"]
    low  = C[C["bucket"] == "low"]

    take = lambda df, n: (df.sample(n=min(n, len(df)), random_state=seed) if not df.empty else df)
    s_high, s_mid, s_low = take(high, n_pos), take(mid, n_corner), take(low, n_neg)

    need_more = n_total - (len(s_high) + len(s_mid) + len(s_low))
    if need_more > 0:
        rest = C.drop(s_high.index.union(s_mid.index).union(s_low.index))
        out = pd.concat([s_high, s_mid, s_low, rest.sample(n=min(need_more, len(rest)), random_state=seed)], ignore_index=True)
    else:
        out = pd.concat([s_high, s_mid, s_low], ignore_index=True)

    out.insert(0, "label", "")  # to be filled manually (1=match, 0=non-match)

    # SIDE-BY-SIDE columns for easy comparison
    side_by_side_cols = [
        # meta & scoring
        "label","pair","score","bucket","rule","geo_m","sim_name","sim_cat",
        # IDs (generic + native)
        "id_a","id_b",
        f"{id_col_a}_a" if id_col_a else None,
        f"{id_col_b}_b" if id_col_b else None,
        # Sources
        "source_a","source_b",
        # Names
        "name_a","name_b",
        "name_norm_a","name_norm_b",
        # Address line 1
        "address_line1_a","address_line1_b",
        # Street & house number
        "street_a","street_b",
        "house_number_a","house_number_b",
        # City/State/ZIP
        "city_a","city_b",
        "state_a","state_b",
        "postal_code_a","postal_code_b",
        # Geo
        "latitude_a","latitude_b",
        "longitude_a","longitude_b",
        # Categories
        "categories_a","categories_b",
        # Phones (may be None for a dataset)
        "phone_e164_a","phone_e164_b",
        "phone_raw_a","phone_raw_b",
    ]
    side_by_side_cols = [c for c in side_by_side_cols if c is not None]

    # Ensure all requested columns exist
    for k in side_by_side_cols:
        if k not in out.columns:
            out[k] = None

    # Reorder
    out = out[side_by_side_cols].reset_index(drop=True)
    return out

def stratified_val_test_split(labeled_df: pd.DataFrame, val_frac=0.5, seed=42):
    """
    Split a manually labeled sheet (requires 'label' in {0,1}) into
    validation and test sets with stratification by label.
    """
    if "label" not in labeled_df.columns:
        raise ValueError("Labeled dataframe must contain a 'label' column with 0/1.")
    L = labeled_df.copy()
    L["label"] = L["label"].map(lambda x: 1 if str(x).strip() in {"1","true","True"} else 0)

    val_parts, test_parts = [], []
    for y in [0,1]:
        grp = L[L["label"] == y]
        if grp.empty: continue
        n_val = int(round(len(grp) * val_frac))
        val_idx = grp.sample(n=n_val, random_state=seed).index
        val_parts.append(L.loc[val_idx])
        test_parts.append(L.drop(index=val_idx))

    val_df  = pd.concat(val_parts,  ignore_index=True) if val_parts  else pd.DataFrame(columns=L.columns)
    test_df = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame(columns=L.columns)
    return val_df, test_df

# ==========================================
# How to run (example)
# ==========================================
# Example:
# cands_12 = build_candidates_pair(
#     df1_380k_clean, df2_uber_eats_clean,
#     pair_name="df1_vs_df2",
#     id_col_a="kaggle380k_id",   # df1 id column
#     id_col_b="uber_eats_id"     # df2 id column
# )
# sheet_12 = sample_for_manual_labeling(
#     cands_12, n_total=800, seed=42,
#     id_col_a="kaggle380k_id", id_col_b="uber_eats_id"
# )
# sheet_12.to_csv("ground_truth_to_label_df1_df2.csv", index=False)
#
# cands_13 = build_candidates_pair(
#     df1_380k_clean, df3_yelp_clean,
#     pair_name="df1_vs_df3",
#     id_col_a="kaggle380k_id",   # df1 id column
#     id_col_b="yelp_id"          # df3 id column
# )
# sheet_13 = sample_for_manual_labeling(
#     cands_13, n_total=800, seed=42,
#     id_col_a="kaggle380k_id", id_col_b="yelp_id"
# )
# sheet_13.to_csv("ground_truth_to_label_df1_df3.csv", index=False)


In [35]:
# -------- Pair 1: df1 (Kaggle 380k) vs df2 (UberEats) --------
cands_12 = build_candidates_pair(
    kaggle380k, uber_eats,
    pair_name="df1_vs_df2",
    id_col_a="kaggle380k_id",   # df1 id column
    id_col_b="uber_eats_id"     # df2 id column
)
sheet_12 = sample_for_manual_labeling(
    cands_12, n_total=1000, seed=42,
    id_col_a="kaggle380k_id", id_col_b="uber_eats_id"
)
sheet_12.to_csv("ground_truth_to_label_df1_df2.csv", index=False)

In [36]:
# -------- Pair 2: df1 (Kaggle 380k) vs df3 (yelp) --------
cands_12 = build_candidates_pair(
    kaggle380k, yelp,
    pair_name="df1_vs_df3",
    id_col_a="kaggle380k_id",   # df1 id column
    id_col_b="yelp_id"     # df2 id column
)
sheet_12 = sample_for_manual_labeling(
    cands_12, n_total=1000, seed=42,
    id_col_a="kaggle380k_id", id_col_b="yelp_id"
)
sheet_12.to_csv("ground_truth_to_label_df1_df3.csv", index=False)

In [37]:
def export_kaggle_uber_label_triplets(input_csv: str, output_csv: str):
    """
    Read the labeling sheet produced for df1 vs df2 and write a CSV (no header)
    containing only: kaggle_id, uber_eats_id, label.
    """
    df = pd.read_csv(input_csv)

    # Find Kaggle and UberEats id columns (prefer native suffixed; fallback to generic id_a/id_b)
    kaggle_col = None
    for c in ["id_a"]:
        if c in df.columns:
            kaggle_col = c; break
    if kaggle_col is None:
        raise ValueError("Could not locate Kaggle ID column (expected 'kaggle380k_id_a' or 'id_a').")

    uber_col = None
    for c in ["id_b"]:
        if c in df.columns:
            uber_col = c; break
    if uber_col is None:
        raise ValueError("Could not locate Uber Eats ID column (expected 'uber_eats_id_b' or 'id_b').")

    if "label" not in df.columns:
        raise ValueError("Input file must contain a 'label' column.")

    out = df[[kaggle_col, uber_col, "label"]].copy()
    out.columns = ["kaggle_id", "uber_eats_id", "label"]

    # Write WITHOUT column headers
    out.to_csv(output_csv, index=False, header=False)


In [38]:
export_kaggle_uber_label_triplets(
    input_csv="ground_truth_to_label_df1_df2.csv",
    output_csv="ground_truth_to_label_df1_df2_labled.csv"
)

In [39]:
export_kaggle_uber_label_triplets(
    input_csv="ground_truth_to_label_df1_df3.csv",
    output_csv="ground_truth_to_label_df1_df3_labled.csv"
)

# Split into Training and Test

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the labeled dataset
df = pd.read_csv("ground_truth_to_label_df1_df2_labled.csv")

# Split into train/test (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save to new CSVs
train_df.to_csv("ground_truth_df1_df2_train.csv", index=False)
test_df.to_csv("ground_truth_df1_df2_test.csv", index=False)

# Load the labeled dataset
df = pd.read_csv("ground_truth_to_label_df1_df3_labled.csv")

# Split into train/test (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save to new CSVs
train_df.to_csv("ground_truth_df1_df3_train.csv", index=False)
test_df.to_csv("ground_truth_df1_df3_test.csv", index=False)



In [7]:
import pandas as pd

# Load the CSV file
input_file = "ml-datasets/ground_truth_df1_df2_test.csv"
output_file = "ml-datasets/ground_truth_df1_df2_test_numeric.csv"

# Read the data
df = pd.read_csv(input_file, names=['id1', 'id2', 'label'],)

# Convert label column from string to numeric (case-insensitive)
df['label'] = df['label'].astype(str).str.lower().map({'true': '1', 'false': '0'})

# Save the transformed DataFrame back to CSV
df.to_csv(output_file, index=False)